In [ ]:
!apt-get update -qq && apt-get install -y -qq sox libsox-fmt-all
!pip install -q manim edge-tts

In [ ]:
import os, json, glob, subprocess
beats = [
    "Xin chào các em học sinh thân mến. Hôm nay, thầy sẽ hướng dẫn các em đếm số phần tử bằng biểu đồ Ven và nguyên lý bù trừ.",
    "Xét hai tập hợp A và B được biểu diễn bằng hai hình tròn.",
    "Phần giao chuẩn xác ở giữa chính là tập hợp A giao B.",
    "Khi ta cộng toàn bộ phần tử của A với toàn bộ phần tử của B,",
    "thì vùng A giao B đã được cộng hai lần.",
    "Do đó, ta phải trừ đi một lần phần giao để vùng này chỉ được tính một lần. Ta có công thức như trên màn hình.",
    "Mở rộng cho ba tập hợp A, B và C.",
    "Khi ta cộng số phần tử của ba tập riêng biệt là A cộng B cộng C,",
    "vùng lõi trung tâm A giao B giao C đã được cộng ba lần.",
    "Tiếp theo, ta trừ đi các phần giao đôi một: A giao B, B giao C và C giao A.",
    "Lúc này, vùng lõi trung tâm lại bị trừ đi đúng ba lần, dẫn đến việc nó chưa được tính lần nào!",
    "Vì vậy, ta bắt buộc phải cộng bù lại một lần phần giao của cả ba tập hợp. Đây chính là nguyên lý bù trừ kinh điển.",
    "Ta cùng áp dụng vào bài toán: Lớp 10A có 20 bạn giỏi Toán, 15 giỏi Văn, 12 giỏi Anh.",
    "Có 6 bạn giỏi Toán và Văn, 5 bạn giỏi Văn và Anh, 4 bạn giỏi Toán và Anh, và 2 bạn giỏi cả 3 môn.",
    "Theo nguyên tắc biểu đồ Ven, ta điền từ trong ra ngoài: Điền số 2 vào phần giao của cả 3 môn trước.",
    "Số bạn chỉ giỏi Toán và Văn là 6 trừ 2 bằng 4 bạn. Tương tự, ta tính được các phần giao còn lại là 3 và 2.",
    "Số bạn chỉ giỏi Toán là 20 trừ đi 4, trừ 2, trừ 2, ta được 12 bạn. Làm tương tự, ta được 6 bạn chỉ giỏi Văn và 5 bạn chỉ giỏi Anh.",
    "Cộng tất cả các vùng độc lập trên biểu đồ, ta được tổng số học sinh giỏi ít nhất một môn là 34 học sinh. Rất trực quan phải không nào!",
    "Thầy chúc các em học tập thật tốt và hẹn gặp lại các em trong các bài giảng tiếp theo!"
]
durations, audio_files = [], []
for idx, text in enumerate(beats):
    txt_f = f'b_{idx}.txt'
    aud_f = f'b_{idx}.mp3'
    with open(txt_f, 'w', encoding='utf-8') as f: f.write(text)
    os.system(f'edge-tts --voice vi-VN-NamMinhNeural --file {txt_f} --write-media {aud_f}')
    cmd = f'ffprobe -v error -show_entries format=duration -of default=noprint_wrappers=1:nokey=1 {aud_f}'
    dur = float(subprocess.check_output(cmd, shell=True).decode().strip())
    durations.append(dur)
    audio_files.append(aud_f)
with open('beat_durations.json', 'w') as f: json.dump(durations, f)
with open('audio_list.txt', 'w') as f: f.write('\n'.join([f"file '{f}'" for f in audio_files]))
subprocess.run('ffmpeg -y -f concat -safe 0 -i audio_list.txt -c copy full_synced_speech.mp3', shell=True)


In [ ]:
manim_code = r'''from manim import *
import json
with open("beat_durations.json") as f: d = json.load(f)
def sync_wait(scene, anim_time, target_time):
    rem = target_time - anim_time
    if rem > 0.05: scene.wait(rem)
class VennMasterSynced(Scene):
    def construct(self):
        title = Text("BIỂU ĐỒ VENN & NGUYÊN LÝ BÙ TRỪ", font_size=36, color=GREEN_B).to_edge(UP, buff=0.8)
        self.play(Write(title), run_time=2.5)
        sync_wait(self, 2.5 + 0.5, d[0])
        self.play(FadeOut(title), run_time=0.5)
        cA2 = Circle(radius=2.0, color=BLUE).shift(LEFT * 1.5)
        cB2 = Circle(radius=2.0, color=RED).shift(RIGHT * 1.5)
        lA2 = Text("A", color=BLUE_B, font_size=40).next_to(cA2, UP+LEFT, buff=-0.2)
        lB2 = Text("B", color=RED_B, font_size=40).next_to(cB2, UP+RIGHT, buff=-0.2)
        self.play(Create(cA2), Create(cB2), Write(lA2), Write(lB2), run_time=2.0)
        sync_wait(self, 2.0, d[1])
        inter2 = Intersection(cA2, cB2, color=GOLD, fill_opacity=0.8, stroke_width=2)
        lCap = Text("A ∩ B", color=BLACK, font_size=28, weight=BOLD).move_to(inter2.get_center())
        self.play(FadeIn(inter2), Write(lCap), run_time=1.5)
        sync_wait(self, 1.5, d[2])
        self.play(cA2.animate.set_fill(BLUE, 0.3), cB2.animate.set_fill(RED, 0.3), run_time=2.0)
        sync_wait(self, 2.0, d[3])
        warn2 = Text("Bị cộng 2 lần!", font_size=24, color=YELLOW, weight=BOLD).next_to(inter2, UP, buff=0.1)
        self.play(Indicate(inter2, color=RED), FadeIn(warn2), run_time=1.5)
        sync_wait(self, 1.5, d[4])
        f2 = Text("|A ∪ B| = |A| + |B| - |A ∩ B|", font_size=34, color=GREEN_B).to_edge(DOWN, buff=0.6)
        self.play(Write(f2), FadeOut(warn2), inter2.animate.set_color(GREEN).set_fill(opacity=0.5), run_time=2.5)
        sync_wait(self, 2.5 + 1.0, d[5])
        self.play(FadeOut(Group(cA2, cB2, lA2, lB2, inter2, lCap, f2)), run_time=1.0)
        cA = Circle(radius=1.8, color=BLUE).shift(UP * 1.0)
        cB = Circle(radius=1.8, color=RED).shift(DOWN * 0.8 + LEFT * 1.2)
        cC = Circle(radius=1.8, color=GREEN).shift(DOWN * 0.8 + RIGHT * 1.2)
        lA = Text("A", color=BLUE_B, font_size=32).next_to(cA, UP, buff=0.1)
        lB = Text("B", color=RED_B, font_size=32).next_to(cB, LEFT, buff=0.1)
        lC = Text("C", color=GREEN_B, font_size=32).next_to(cC, RIGHT, buff=0.1)
        self.play(Create(cA), Create(cB), Create(cC), Write(lA), Write(lB), Write(lC), run_time=2.5)
        sync_wait(self, 2.5, d[6])
        f3_1 = Text("|A ∪ B ∪ C| = |A| + |B| + |C|", font_size=26, color=WHITE).to_edge(DOWN, buff=0.5)
        self.play(cA.animate.set_fill(BLUE, 0.25), cB.animate.set_fill(RED, 0.25), cC.animate.set_fill(GREEN, 0.25), Write(f3_1), run_time=2.5)
        sync_wait(self, 2.5, d[7])
        core = Intersection(Intersection(cA, cB), cC, color=YELLOW, fill_opacity=0.9, stroke_width=2)
        warn3_1 = Text("Cộng 3 lần!", font_size=20, color=YELLOW, weight=BOLD).next_to(core, UP, buff=0.0)
        self.play(FadeIn(core), FadeIn(warn3_1), run_time=1.5)
        self.play(Indicate(core, scale_factor=1.5), run_time=1.5)
        sync_wait(self, 3.0, d[8])
        iAB = Intersection(cA, cB, color=RED, fill_opacity=0.6, stroke_width=2)
        iBC = Intersection(cB, cC, color=RED, fill_opacity=0.6, stroke_width=2)
        iCA = Intersection(cC, cA, color=RED, fill_opacity=0.6, stroke_width=2)
        f3_2 = Text("|A ∪ B ∪ C| = |A| + |B| + |C| - (|A ∩ B| + |B ∩ C| + |C ∩ A|)", font_size=24, color=RED_A).to_edge(DOWN, buff=0.5)
        self.play(FadeIn(iAB), FadeIn(iBC), FadeIn(iCA), Transform(f3_1, f3_2), FadeOut(warn3_1), run_time=2.5)
        sync_wait(self, 2.5, d[9])
        warn3_2 = Text("Bị trừ 3 lần -> Bằng 0!", font_size=20, color=RED, weight=BOLD).next_to(core, UP, buff=0.0)
        self.play(FadeIn(warn3_2), core.animate.set_fill(BLACK, 1.0), run_time=2.0)
        self.play(Indicate(core, color=RED, scale_factor=1.5), run_time=1.5)
        sync_wait(self, 3.5, d[10])
        f3_3 = Text("|A ∪ B ∪ C| = |A| + |B| + |C| - (|A ∩ B| + |B ∩ C| + |C ∩ A|) + |A ∩ B ∩ C|", font_size=22, color=YELLOW_B).to_edge(DOWN, buff=0.5)
        self.play(Transform(f3_1, f3_3), core.animate.set_fill(GOLD, 1.0), FadeOut(warn3_2), run_time=2.5)
        sync_wait(self, 2.5 + 1.0, d[11])
        self.play(FadeOut(Group(cA, cB, cC, lA, lB, lC, iAB, iBC, iCA, core, f3_1)), run_time=1.0)
        t4 = Text("BÀI TOÁN LỚP 10A", font_size=32, color=GOLD).to_edge(UP, buff=0.2)
        q1 = Text("20 Toán, 15 Văn, 12 Anh", font_size=24, color=WHITE).next_to(t4, DOWN, buff=0.2)
        cT = Circle(radius=1.7, color=BLUE).shift(DOWN * 0.2 + LEFT * 2.5)
        cV = Circle(radius=1.7, color=RED).shift(DOWN * 1.9 + LEFT * 3.5)
        cE = Circle(radius=1.7, color=GREEN).shift(DOWN * 1.9 + LEFT * 1.5)
        lT = Text("Toán", font_size=26, color=BLUE_B).next_to(cT, UP, buff=0.1)
        lV = Text("Văn", font_size=26, color=RED_B).next_to(cV, LEFT, buff=0.1)
        lE = Text("Anh", font_size=26, color=GREEN_B).next_to(cE, RIGHT, buff=0.1)
        self.play(Write(t4), Write(q1), Create(cT), Create(cV), Create(cE), Write(lT), Write(lV), Write(lE), run_time=3.0)
        sync_wait(self, 3.0, d[12])
        q2 = Text("Giao 2 môn: 6 Toán-Văn, 5 Văn-Anh, 4 Toán-Anh", font_size=24, color=WHITE).next_to(q1, DOWN, buff=0.15)
        q3 = Text("Giao 3 môn: 2 bạn. (Hỏi giỏi ít nhất 1 môn?)", font_size=24, color=YELLOW).next_to(q2, DOWN, buff=0.15)
        self.play(Write(q2), Write(q3), run_time=3.0)
        sync_wait(self, 3.0, d[13])
        core_P = Intersection(Intersection(cT, cV), cE, color=YELLOW, fill_opacity=0.8, stroke_width=2)
        nCore = Text("2", font_size=26, weight=BOLD, color=BLACK).move_to(core_P.get_center())
        self.play(FadeIn(core_P), GrowFromCenter(nCore), run_time=2.0)
        sync_wait(self, 2.0, d[14])
        pTV = Difference(Intersection(cT, cV), cE, color=ORANGE, fill_opacity=0.6, stroke_width=2)
        pVE = Difference(Intersection(cV, cE), cT, color=PURPLE, fill_opacity=0.6, stroke_width=2)
        pET = Difference(Intersection(cE, cT), cV, color=TEAL, fill_opacity=0.6, stroke_width=2)
        nTV = Text("4", font_size=26, weight=BOLD).move_to(pTV.get_center())
        nVE = Text("3", font_size=26, weight=BOLD).move_to(pVE.get_center())
        nET = Text("2", font_size=26, weight=BOLD).move_to(pET.get_center())
        self.play(FadeIn(pTV), GrowFromCenter(nTV), FadeIn(pVE), GrowFromCenter(nVE), FadeIn(pET), GrowFromCenter(nET), run_time=3.0)
        sync_wait(self, 3.0, d[15])
        oT = Difference(Difference(cT, cV), cE, color=BLUE, fill_opacity=0.4, stroke_width=2)
        oV = Difference(Difference(cV, cT), cE, color=RED, fill_opacity=0.4, stroke_width=2)
        oE = Difference(Difference(cE, cT), cV, color=GREEN, fill_opacity=0.4, stroke_width=2)
        nOT = Text("12", font_size=26, weight=BOLD).move_to(oT.get_center())
        nOV = Text("6", font_size=26, weight=BOLD).move_to(oV.get_center())
        nOE = Text("5", font_size=26, weight=BOLD).move_to(oE.get_center())
        self.play(FadeIn(oT), GrowFromCenter(nOT), FadeIn(oV), GrowFromCenter(nOV), FadeIn(oE), GrowFromCenter(nOE), run_time=3.0)
        sync_wait(self, 3.0, d[16])
        ans_box = Rectangle(width=5.5, height=2.0, color=WHITE, fill_color=BLACK, fill_opacity=0.8).shift(RIGHT * 3.5 + DOWN * 1.0)
        ans_t1 = Text("Tổng số học sinh:", font_size=24, color=WHITE).next_to(ans_box.get_top(), DOWN, buff=0.2)
        ans_eq = Text("12+6+5 + 4+3+2 + 2", font_size=24, color=YELLOW).next_to(ans_t1, DOWN, buff=0.2)
        ans_res = Text("= 34", font_size=36, color=GREEN_B, weight=BOLD).next_to(ans_eq, DOWN, buff=0.2)
        self.play(Create(ans_box), Write(ans_t1), Write(ans_eq), run_time=2.5)
        self.play(Write(ans_res), Indicate(ans_res, scale_factor=1.3), run_time=2.0)
        sync_wait(self, 4.5 + 1.0, d[17])
        self.play(FadeOut(Group(t4, q1, q2, q3, cT, cV, cE, lT, lV, lE, core_P, nCore, pTV, nTV, pVE, nVE, pET, nET, oT, nOT, oV, nOV, oE, nOE, ans_box, ans_t1, ans_eq, ans_res)), run_time=1.0)
        outro = Text("CHÚC CÁC EM HỌC TẬP THẬT TỐT!", font_size=40, color=YELLOW, weight=BOLD)
        self.play(Write(outro), run_time=2.0)
        self.play(Circumscribe(outro, color=GOLD), run_time=2.0)
        sync_wait(self, 4.0, d[18])
'''
with open('venn_synced.py', 'w', encoding='utf-8') as f: f.write(manim_code)
subprocess.run('manim -qm venn_synced.py VennMasterSynced -o raw_synced.mp4', shell=True)


In [ ]:
raw_videos = glob.glob('media/**/raw_synced.mp4', recursive=True)
if raw_videos:
    subprocess.run(f'ffmpeg -y -i "{raw_videos[0]}" -i full_synced_speech.mp3 -c:v copy -c:a aac -shortest BaiGiang_BieuDoVenn.mp4', shell=True)
    print("Render success: BaiGiang_BieuDoVenn.mp4")
